<img src="https://res.cloudinary.com/dtizipxds/image/upload/q_auto/f_auto/v1776264397/banner_yvwajv.png" width="100%">


In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn umap-learn


# Solution - Dimensionality Reduction


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap

sns.set_theme(style="whitegrid")
full_df = pd.read_csv(Path("data.csv"))
sol_df = pd.read_csv(Path("solution.csv"))
X = full_df[["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]]
y = full_df["species"]
X_scaled = StandardScaler().fit_transform(X)


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

tsne = TSNE(n_components=2, random_state=42, perplexity=20)
X_tsne = tsne.fit_transform(X_scaled)

um = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.2)
X_umap = um.fit_transform(X_scaled)

embed_df = pd.DataFrame({
    "id": full_df["id"],
    "species": y,
    "pca_1": X_pca[:,0],
    "pca_2": X_pca[:,1],
    "tsne_1": X_tsne[:,0],
    "tsne_2": X_tsne[:,1],
    "umap_1": X_umap[:,0],
    "umap_2": X_umap[:,1],
})
embed_df.head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))
sns.scatterplot(data=embed_df, x="pca_1", y="pca_2", hue="species", s=70, ax=axes[0])
axes[0].set_title("PCA")
sns.scatterplot(data=embed_df, x="tsne_1", y="tsne_2", hue="species", s=70, ax=axes[1], legend=False)
axes[1].set_title("t-SNE")
sns.scatterplot(data=embed_df, x="umap_1", y="umap_2", hue="species", s=70, ax=axes[2], legend=False)
axes[2].set_title("UMAP")
plt.tight_layout()
plt.show()


In [ ]:
# Check PCA reference values for test ids
check = embed_df.merge(sol_df[["id", "pca_1", "pca_2"]], on="id", suffixes=("_new", "_ref"))
mae = (check[["pca_1_new", "pca_1_ref", "pca_2_new", "pca_2_ref"]]
       .assign(e1=lambda d: (d["pca_1_new"] - d["pca_1_ref"]).abs(),
               e2=lambda d: (d["pca_2_new"] - d["pca_2_ref"]).abs())[["e1", "e2"]].mean())
print("PCA MAE vs reference:")
print(mae)
